<a href="https://colab.research.google.com/github/seeker91/Predicting-net-rents-per-square-foot/blob/main/Predicting_net_rents_per_square_foot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#import all the relevant packages

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt

In [2]:
#decide on the number of samples

n_samples = 1000

In [3]:
#create a dummy dataset for the Canadian real estate market

np.random.seed(42)
rng = np.random.default_rng(seed = 42)
real_data = {"postal_code": np.random.choice(["M5V", "V6B", "H3A", "T2P", "K1P"], n_samples),
             "b_class": np.random.choice(["A", "B", "C"], n_samples, p = [0.4, 0.4, 0.2]),
             "b_size": np.random.randint(1000, 50000, n_samples),
             "lease_months": np.random.choice([36, 60, 120], n_samples, p = [0.2, 0.6, 0.2]),
             "add_rent_sq_ft": rng.uniform(10.0, 25.0, n_samples)}

In [7]:
df = pd.DataFrame(real_data)
df.head()

,postal_code,b_class,b_size,lease_months,add_rent_sq_ft
0,T2P,C,37224,60,21.609341
1,K1P,B,8788,60,16.583177
2,H3A,B,29082,60,22.878969
3,K1P,C,25034,60,20.460520
4,K1P,B,36479,120,11.412660


In [13]:
#Estimating rent based on base rents

np.random.seed(42)
location_base = df["postal_code"].map({"M5V": 35, "V6B": 38, "H3A": 34, "T2P": 30, "K1P": 27})
class_base = df["b_class"].map({"A": 15, "B": 5, "C": 0})
size_discount = (df["b_size"] / 10000) * -0.5

df["net_rent"] = location_base + class_base + size_discount + np.random.normal(0, 3, n_samples)
df.head()

,postal_code,b_class,b_size,lease_months,add_rent_sq_ft,net_rent
0,T2P,C,37224,60,21.609341,29.628942
1,K1P,B,8788,60,16.583177,31.145807
2,H3A,B,29082,60,22.878969,39.488966
3,K1P,C,25034,60,20.460520,30.317390
4,K1P,B,36479,120,11.412660,29.473590


In [14]:
#Now we have the target variable, we can start with modeling

X = df.copy()
X.drop("net_rent", axis = 1, inplace = True)
y = df["net_rent"]

In [15]:
#instantiate the model

from sklearn.model_selection import GridSearchCV
rf = RandomForestRegressor(random_state = 42)

cv_params = {"n_estimators": [100, 200, 300], "max_depth": [2, 3, 5, None], "min_samples_split": [2, 3, 4], "min_samples_leaf": [1, 3, 4]}
rf_cv = GridSearchCV(rf, cv_params, scoring = "neg_mean_squared_error", cv = 5, n_jobs = -1)

In [18]:
#Split the data

X = pd.get_dummies(X, columns = ["postal_code", "b_class"], drop_first = True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 42)

In [19]:
%%time
rf_cv.fit(X_train, y_train)

CPU times: user 2.27 s, sys: 278 ms, total: 2.55 s
Wall time: 3min 49s


GridSearchCV(cv=5, estimator=RandomForestRegressor(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [2, 3, 5, None],
                         'min_samples_leaf': [1, 3, 4],
                         'min_samples_split': [2, 3, 4],
                         'n_estimators': [100, 200, 300]},
             scoring='neg_mean_squared_error')

In [20]:
rf_cv.best_params_

{'max_depth': 5,
 'min_samples_leaf': 4,
 'min_samples_split': 2,
 'n_estimators': 200}

In [21]:
#Fit with the best hyperparameters
rf_best = RandomForestRegressor(max_depth = 5, min_samples_leaf = 4, min_samples_split = 2, n_estimators = 200, random_state = 42)
rf_best.fit(X_train, y_train)

RandomForestRegressor(max_depth=5, min_samples_leaf=4, n_estimators=200,
                      random_state=42)

In [22]:
#Calculate predicted values from test predictors

y_pred = rf_best.predict(X_test)

In [23]:
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"Mean Absolute Error: {mae}")
print(f"R2 Score: {r2}")

Mean Absolute Error: 2.452500178610882
R2 Score: 0.8432406174479418


In [33]:
importances = rf_best.feature_importances_
data_important = {"Features": X.columns, "Importance": importances}
df_important = pd.DataFrame(data_important)
df_important.sort_values(by = "Importance", ascending = False)

,Features,Importance
7,b_class_B,0.384337
8,b_class_C,0.289459
3,postal_code_K1P,0.168378
5,postal_code_T2P,0.076603
6,postal_code_V6B,0.052292
0,b_size,0.018909
2,add_rent_sq_ft,0.007288
1,lease_months,0.001820
4,postal_code_M5V,0.000915


In [27]:
X.columns

Index(['b_size', 'lease_months', 'add_rent_sq_ft', 'postal_code_K1P',
       'postal_code_M5V', 'postal_code_T2P', 'postal_code_V6B', 'b_class_B',
       'b_class_C'],
      dtype='object')

In [35]:
# Example: Predicting rent for a brand new property entry
import warnings
warnings.filterwarnings('ignore')

new_lease_features = [[45000, 36, 12.50, 0, 0, 0, 0, 1, 0]]  #Class B building
predicted_price = rf_best.predict(new_lease_features)
print(f"Predicted Net Rent: ${predicted_price[0]:.2f} / sqft")

Predicted Net Rent: $38.30 / sqft
